# setting

In [117]:
import numpy as np
import pandas as pd
import seaborn as sns
from anytree import Node, RenderTree

In [2]:
working_path = '/cluster/raid/home/f80878961/beekeeping/'
data_path = '{}data/'.format(working_path)

In [11]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [75]:
data_fn = '{}raw data 10.3.26_test an Victor.xlsx'.format(data_path)

df = pd.read_excel(data_fn, sheet_name='Raw data', engine='openpyxl')

/cluster/raid/home/f80878961/.conda/envs/planb/lib/python3.9/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


# Pedigree visu

In [112]:
name2node = {}
name2root = {}

for ped in sorted(set(df['Pedigree'].dropna().to_list()), key=lambda x: len(x), reverse=False):
    ped = ped.rstrip()
    s_ped = ped.split('-')

    # root --> new ped
    if len(s_ped) == 1:
        root = Node(ped, parent=None, event='Root')
        name2node[ped] = root
        name2root[ped] = root
        continue

    parent = name2node['-'.join(s_ped[:-1])]
    
    # frutling
    if s_ped[-1].startswith('F'):
        name2node[ped] = Node(ped, parent=parent, event='Frütling')

    # brutling
    elif s_ped[-1].startswith('B'):
         name2node[ped] = Node(ped, parent=parent, event='Brutling')

    # requeen
    elif s_ped[-1].startswith('R'):
         name2node[ped] = Node(ped, parent=parent, event='Requeening')


In [114]:
for root in name2root.values():
    for pre, fill, node in RenderTree(root):
        print(f"{pre}{node.name} -> {node.event}")


x24.244 -> Root
x24.203 -> Root
x24.252 -> Root
x24.231 -> Root
├── x24.231-B25 -> Brutling
└── x24.231-F25 -> Frütling
    └── x24.231-F25-R25 -> Requeening
x24.159 -> Root
├── x24.159-F25 -> Frütling
│   └── x24.159-F25-R25 -> Requeening
└── x24.159-B25 -> Brutling
x24.215 -> Root
├── x24.215-F25 -> Frütling
└── x24.215-B25 -> Brutling
x24.116 -> Root
x24.490 -> Root
x24.130 -> Root
└── x24.130-F25 -> Frütling
x24.111 -> Root
x24.255 -> Root
└── x24.255-R25 -> Requeening
x24.146 -> Root
x24.160 -> Root
x24.208 -> Root
├── x24.208-F25 -> Frütling
└── x24.208-B25 -> Brutling
    └── x24.208-B25-R25 -> Requeening
x24.225 -> Root
├── x24.225-F25 -> Frütling
└── x24.225-B25 -> Brutling
    └── x24.225-B25-R25 -> Requeening
x24.224 -> Root
x24.117 -> Root
x24.216 -> Root
x24.170 -> Root
x24.230 -> Root
├── x24.230-B25 -> Brutling
└── x24.230-F25 -> Frütling
x24.479 -> Root
x24.202 -> Root
x24.185 -> Root
└── x24.185-R25 -> Requeening
x24.119 -> Root
x24.167 -> Root
├── x24.167-B25 -> Brutl

In [89]:

from anytree import Node, RenderTree

root = Node("Root", event="Started system")
a = Node("A", parent=root, event="Checked")
b = Node("B", parent=root, event="Synced")
c = Node("C", parent=a, event="Completed")

for pre, fill, node in RenderTree(root):
    print(f"{pre}{node.name} -> {node.event}")


Root -> Started system
├── A -> Checked
│   └── C -> Completed
└── B -> Synced


In [115]:
from anytree.exporter import DotExporter

DotExporter(name2root['x24.132'],
            nodeattrfunc=lambda n: f'label="{n.name}\n{n.event}"'
).to_picture("{}tree.png".format(data_path))


# Data Validation 

data validation

1. structure : check columns and datatypes
2. quality / errors : outliers and duplicate, allowed categories and ranges
3. logics: e.g., pedigree ideas


https://towardsdatascience.com/data-validation-with-pandera-in-python-f07b0f845040/

In [67]:
import pandera.pandas as pa

In [68]:
test_df = df[['Apiary']] #, 'VAR varroa/day']]

In [69]:
test_df.loc[:, "Apiary"]= test_df['Apiary'].astype('string')

In [70]:
#test_df.loc[:, "Date"] = pd.to_datetime(test_df["Date"])

In [71]:
class PlanbModel(pa.DataFrameModel):
    Apiary: str

In [73]:
PlanbModel.validate(test_df)

,Apiary
0,AB
1,AB
2,AB
3,AB
4,AB
...,...
2778,HE
2779,HE
2780,HE
2781,HE
